# Layout crop (batch) — header/footer off, not redacted

Opens from GitHub **without** `src/`. **Runtime → Run all.** First code cell must print `BOOTSTRAP_V6` and `tick_policy: slash-v2`. Branch **`block1`**.

Edit **KEEP_TYPES / DROP_TYPES / TOP / BOTTOM** in the config cell. That layout applies to **every** image in the batch.

**Do not upload clinic PHI to Colab** if you can crop locally first (`python -m med_doc.privacy photos/ --out cropped/`).


## 0. Download src/med_doc (zipball)


In [ ]:
# BOOTSTRAP_V6 — always refresh zipball (stale /content/epq3 lacks new kwargs like output_mode)
import importlib
import inspect
import os
import shutil
import sys
import urllib.request
import zipfile
from pathlib import Path

CONTENT = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = CONTENT / "epq3"
SRC = REPO / "src"
URL = "https://codeload.github.com/RwaRwa599/epq3/zip/refs/heads/block1"

zpath = CONTENT / "epq3-block1.zip"
print("Downloading", URL)
urllib.request.urlretrieve(URL, zpath)
extract = CONTENT / "_epq3_extract"
if extract.exists():
    shutil.rmtree(extract)
extract.mkdir()
with zipfile.ZipFile(zpath) as zf:
    zf.extractall(extract)
found = list(extract.glob("*/src/med_doc/__init__.py"))
if not found:
    raise RuntimeError(f"zip missing src/med_doc: {list(extract.iterdir())}")
unpacked = found[0].parents[2]
if REPO.exists():
    shutil.rmtree(REPO)
shutil.move(str(unpacked), str(REPO))
shutil.rmtree(extract, ignore_errors=True)
zpath.unlink(missing_ok=True)

src = str(SRC.resolve())
while src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
os.chdir(REPO)
for name in list(sys.modules):
    if name == "med_doc" or name.startswith("med_doc."):
        del sys.modules[name]
importlib.invalidate_caches()
import med_doc
from med_doc.pipeline import run_blocks_1_to_5

print("BOOTSTRAP_V6")
print("cwd:", os.getcwd())
print("med_doc:", med_doc.__file__)
print("params:", list(inspect.signature(run_blocks_1_to_5).parameters))
from med_doc.htr.marks import TICK_POLICY
print("tick_policy:", TICK_POLICY)
if "output_mode" not in inspect.signature(run_blocks_1_to_5).parameters:
    raise RuntimeError(
        "stale med_doc (no output_mode). Runtime → Disconnect and delete runtime, "
        "re-open Run_in_Colab.ipynb from GitHub branch block1, then Run all."
    )
if TICK_POLICY != "slash-v2":
    raise RuntimeError(
        f"stale med_doc tick_policy={TICK_POLICY!r}. Disconnect and delete runtime, "
        "re-open Run_in_Colab.ipynb from GitHub branch block1."
    )


In [ ]:
# Runtime deps via pip CLI (not %pip / not pip -e — those restart Colab mid-run).
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "pydantic", "matplotlib", "Pillow", "numpy"]
)


## 1. Set the crop layout for this batch

`backend="template"` uses fractions of the page (default drops v1 header ~0–0.08 and footer ~0.90–1).

`layoutparser_then_template` uses [LayoutParser](https://github.com/Layout-Parser/layout-parser) PubLayNet types, then the same band if nothing is detected.


In [ ]:
# Runtime → Disconnect and delete runtime if this import fails.
from med_doc.privacy import CropConfig, crop_batch, load_crop_config

cfg = load_crop_config()  # configs/layout_crop.json

# --- what LayoutParser is allowed to keep / must drop ---
cfg.keep_types = ["Table", "List"]
cfg.drop_types = ["Title", "Figure", "Text"]
cfg.combine = "largest"  # the column grid, not header+columns stacked
cfg.score_threshold = 0.5

# --- template band: drop name header only (v1 y=0.08–1.0) ---
cfg.template.top = 0.08     # start at Clinical Information (~1cm above columns)
cfg.template.bottom = 1.0   # keep tubes / office footer
cfg.template.left = 0.0
cfg.template.right = 1.0

# "template" | "layoutparser" | "layoutparser_then_template"
cfg.backend = "template"
# cfg.backend = "layoutparser_then_template"  # pip install layoutparser paddlepaddle first

print(cfg.model_dump())


## 2. Batch folder / ZIP / upload → cropped PNGs


In [ ]:
from pathlib import Path
import shutil
from med_doc.privacy import crop_batch

OUT = Path("/content/cropped") if Path("/content").is_dir() else Path("outputs/layout_crop")
OUT.mkdir(parents=True, exist_ok=True)

USE_UPLOAD = False
BATCH_DIR = None  # e.g. Path("/content/photos") or Path("photos.zip")

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    names = list(uploaded.keys())
    if len(names) == 1 and names[0].lower().endswith(".zip"):
        batch_input = names[0]
    else:
        raw = OUT / "_raw"
        raw.mkdir(exist_ok=True)
        for n in names:
            shutil.move(n, raw / n)
        batch_input = raw
elif BATCH_DIR is not None:
    batch_input = BATCH_DIR
else:
    import cv2
    import numpy as np
    demo = OUT / "_demo_raw"
    demo.mkdir(exist_ok=True)
    canvas = np.full((200, 160, 3), 40, np.uint8)
    canvas[:20] = (0, 0, 180)
    canvas[20:176] = (200, 200, 200)
    canvas[176:] = (0, 180, 0)
    cv2.imwrite(str(demo / "demo_a.png"), canvas)
    cv2.imwrite(str(demo / "demo_b.png"), canvas)
    batch_input = demo  # synthetic stripes only - not clinic photos

result = crop_batch(batch_input, OUT, config=cfg)
print("wrote", result["output_dir"], "ok", result["manifest"]["successful"], "/", result["manifest"]["total"])
for row in result["manifest"]["documents"]:
    print(row)


Then run Blocks 1–5 on `OUT` (the cropped folder), not the originals.
